# LangChain + MCP 입문 가이드

## 이 노트북은 무엇을 배우나요?

```
┌─────────────────────────────────────────────────────────┐
│                                                         │
│   여러분의 코드          MCP 서버          OpenAI        │
│   (LangChain)    ←→   (도구 모음)   ←→   (GPT 모델)    │
│                                                         │
│   "15 곱하기 7"    →   multiply(15,7)  →  "105입니다"   │
│                                                         │
└─────────────────────────────────────────────────────────┘
```

### 핵심 개념 3가지

**① MCP 서버** = AI가 사용할 수 있는 도구 창고
- 계산기, 파일 읽기, 날씨 조회 같은 기능들을 모아둔 곳
- 우리가 직접 만들거나 다른 사람이 만든 것을 사용할 수 있어요

**② LangChain** = AI 앱을 만드는 프레임워크
- MCP 서버의 도구를 AI 모델에 연결해줘요

**③ Agent** = 도구를 사용해서 문제를 스스로 해결하는 AI
- 질문을 받으면 → 어떤 도구를 쓸지 판단 → 도구 실행 → 결과 정리

---

### 학습 순서
```
STEP 1. 패키지 설치 & API 키 설정
STEP 2. MCP 서버 만들기 (도구 창고 만들기)
STEP 3. MCP 서버에 뭐가 있는지 확인하기
STEP 4. Agent 만들어서 질문하기
STEP 5. 대화 이어가기 (멀티턴)
STEP 6. 여러 MCP 서버 동시에 사용하기
```

---
## STEP 1. 패키지 설치 및 API 키 설정

### 필요한 패키지 설명
| 패키지 | 역할 |
|---|---|
| `mcp` | MCP 서버를 만들고 연결하는 핵심 라이브러리 |
| `langchain-mcp-adapters` | MCP 도구를 LangChain에서 쓸 수 있게 변환 |
| `langchain-openai` | OpenAI GPT 모델을 LangChain에서 사용 |
| `langgraph` | Agent 로직을 구현하는 라이브러리 |

In [ ]:
# 필요한 패키지를 한 번에 설치합니다
#!pip install mcp langchain-mcp-adapters langchain-openai langgraph

In [ ]:
from dotenv import load_dotenv
import os
# .env 파일을 불러와서 환경 변수로 설정
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
print(OPENAI_API_KEY[:5])

print("✅ API 키 설정 완료")

sk-pr
✅ API 키 설정 완료


---
## STEP 2. MCP 서버 만들기

MCP 서버는 AI가 사용할 수 있는 **도구(Tool)들을 담은 서버**입니다.

아래에서는 간단한 **계산기 MCP 서버**를 만들어봅니다.

```
[ MCP 서버 구조 ]

@mcp.tool()       ← 이 데코레이터를 붙이면 AI가 사용할 수 있는 도구가 됩니다
def add(a, b):    ← 일반 Python 함수처럼 작성
    return a + b  ← 결과를 반환
```

> **💡 핵심**: 함수에 `@mcp.tool()`만 붙이면 끝! AI가 자동으로 이 함수를 도구로 인식합니다.

In [ ]:
# math_server.py 파일로 저장합니다
# (별도 프로세스로 실행되어야 하기 때문에 파일로 만듭니다)

math_server_code = '''
from mcp.server.fastmcp import FastMCP

# MCP 서버 생성 (이름을 붙여줍니다)
mcp = FastMCP("계산기 서버")

# ---- 도구(Tool) 정의 ----
# @mcp.tool() 데코레이터를 붙이면 AI가 사용할 수 있는 도구가 됩니다
# 함수의 docstring(""" """)이 AI에게 전달되는 도구 설명입니다!

@mcp.tool()
def add(a: float, b: float) -> float:
    """두 숫자를 더합니다. 예: add(3, 5) → 8"""
    return a + b

@mcp.tool()
def subtract(a: float, b: float) -> float:
    """두 숫자를 뺍니다. 예: subtract(10, 3) → 7"""
    return a - b

@mcp.tool()
def multiply(a: float, b: float) -> float:
    """두 숫자를 곱합니다. 예: multiply(4, 5) → 20"""
    return a * b

@mcp.tool()
def divide(a: float, b: float) -> float:
    """두 숫자를 나눕니다. 예: divide(10, 2) → 5. 단, b가 0이면 에러!"""
    if b == 0:
        raise ValueError("0으로 나눌 수 없습니다!")
    return a / b

# 서버 실행 (stdio 방식 = 표준 입출력으로 통신)
if __name__ == "__main__":
    mcp.run(transport="stdio")
'''

# 파일로 저장
with open("math_server.py", "w", encoding="utf-8") as f:
    f.write(math_server_code)

print("✅ math_server.py 생성 완료!")
print()
print("📦 이 서버가 제공하는 도구:")
print("  - add(a, b)      : 더하기")
print("  - subtract(a, b) : 빼기")
print("  - multiply(a, b) : 곱하기")
print("  - divide(a, b)   : 나누기")

✅ math_server.py 생성 완료!

📦 이 서버가 제공하는 도구:
  - add(a, b)      : 더하기
  - subtract(a, b) : 빼기
  - multiply(a, b) : 곱하기
  - divide(a, b)   : 나누기


---
## STEP 3. MCP 서버에 연결하고 도구 목록 확인하기

서버를 만들었으니, 실제로 연결해서 어떤 도구들이 있는지 확인해봅시다.

```
[ 연결 흐름 ]

 내 코드
   │
   ├─ stdio_client()  ← 서버 프로세스를 실행하고 파이프로 연결
   │
   └─ ClientSession() ← 연결된 서버와 대화하는 세션
         │
         ├─ session.list_tools()    → 도구 목록 조회
         └─ load_mcp_tools(session) → LangChain 형식으로 변환
```

> **💡 왜 async/await?**  
> MCP 통신은 네트워크처럼 기다림이 발생해요. `async/await`는 기다리는 동안 다른 작업을 할 수 있게 해줍니다.

> **🪟 Windows + Jupyter 주의사항**  
> Jupyter의 `sys.stderr`는 가상 스트림이라 MCP 서버 프로세스에 직접 연결할 수 없어요.  
> 그래서 `errlog=subprocess.DEVNULL`을 명시적으로 전달해 stderr를 버립니다.  
> 이 옵션이 없으면 `UnsupportedOperation: fileno` 에러가 발생합니다!

In [ ]:
import subprocess  # subprocess.DEVNULL 사용을 위해 import
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
from langchain_mcp_adapters.tools import load_mcp_tools

# MCP 서버 실행 정보 설정
# ("python math_server.py" 명령어를 실행하는 것과 같습니다)
server_params = StdioServerParameters(
    command="python",       # 실행할 명령어
    args=["math_server.py"] # 명령어의 인자 (실행할 파일)
)

async def check_available_tools():
    """MCP 서버에 연결해서 사용 가능한 도구를 확인합니다"""

    # ✅ Windows + Jupyter 호환 핵심 포인트!
    # errlog=subprocess.DEVNULL → 서버의 stderr 출력을 그냥 버립니다.
    # 이걸 안 쓰면 Jupyter의 가상 stderr 때문에 fileno 에러 발생!
    async with stdio_client(server_params, errlog=subprocess.DEVNULL) as (read, write):
        # 연결된 서버와 대화할 세션을 만듭니다
        async with ClientSession(read, write) as session:

            # 세션 초기화 (악수하기 - 서로를 인식하는 과정)
            await session.initialize()

            # ① 원시 MCP 도구 목록 조회 (어떤 도구들이 있는지 확인)
            tools_info = await session.list_tools()
            print("📋 MCP 서버의 원시 도구 목록:")
            for tool in tools_info.tools:
                print(f"  🔧 {tool.name}: {tool.description}")

            print()

            # ② LangChain 형식으로 변환 (Agent가 쓸 수 있는 형태로 변환)
            langchain_tools = await load_mcp_tools(session)
            print("🔄 LangChain 형식으로 변환된 도구들:")
            for tool in langchain_tools:
                print(f"  ✅ {tool.name}: {tool.description}")

            print()
            print(f"총 {len(langchain_tools)}개의 도구를 Agent에서 사용할 수 있습니다!")

# Jupyter에서는 await를 직접 사용할 수 있습니다
await check_available_tools()

📋 MCP 서버의 원시 도구 목록:
  🔧 add: 두 숫자를 더합니다. 예: add(3, 5) → 8
  🔧 subtract: 두 숫자를 뺍니다. 예: subtract(10, 3) → 7
  🔧 multiply: 두 숫자를 곱합니다. 예: multiply(4, 5) → 20
  🔧 divide: 두 숫자를 나눕니다. 예: divide(10, 2) → 5. 단, b가 0이면 에러!

🔄 LangChain 형식으로 변환된 도구들:
  ✅ add: 두 숫자를 더합니다. 예: add(3, 5) → 8
  ✅ subtract: 두 숫자를 뺍니다. 예: subtract(10, 3) → 7
  ✅ multiply: 두 숫자를 곱합니다. 예: multiply(4, 5) → 20
  ✅ divide: 두 숫자를 나눕니다. 예: divide(10, 2) → 5. 단, b가 0이면 에러!

총 4개의 도구를 Agent에서 사용할 수 있습니다!


---
## STEP 4. Agent 만들어서 질문하기

이제 GPT 모델 + MCP 도구를 합쳐서 **Agent**를 만들어봅시다!

```
[ Agent 동작 방식 - ReAct 패턴 ]

 사용자: "15 곱하기 7을 계산하고 23을 빼줘"
      │
      ▼
 🤖 GPT가 생각: "곱하기 먼저 해야겠다 → multiply 도구 사용!"
      │
      ▼
 🔧 도구 실행: multiply(15, 7) → 105
      │
      ▼
 🤖 GPT가 생각: "105에서 23을 빼야한다 → subtract 도구 사용!"
      │
      ▼
 🔧 도구 실행: subtract(105, 23) → 82
      │
      ▼
 💬 최종 답변: "15 × 7 = 105, 105 - 23 = 82입니다!"
```

In [ ]:
import subprocess
from langchain_openai import ChatOpenAI
from langchain_mcp_adapters.tools import load_mcp_tools
from langgraph.prebuilt import create_react_agent
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

# GPT 모델 초기화
# temperature=0 → 일관된 답변 (창의성 없이 정확하게)
model = ChatOpenAI(model="gpt-4o-mini", temperature=0)

server_params = StdioServerParameters(
    command="python",
    args=["math_server.py"]
)

async def ask_agent(question: str):
    """Agent에게 질문하고 결과를 보기 좋게 출력합니다"""

    # ✅ errlog=subprocess.DEVNULL : Windows + Jupyter 호환 필수 옵션
    async with stdio_client(server_params, errlog=subprocess.DEVNULL) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()

            # MCP 도구를 LangChain 형식으로 로드
            tools = await load_mcp_tools(session)

            # Agent 생성 (GPT 모델 + MCP 도구들)
            agent = create_react_agent(model, tools)

            print(f"❓ 질문: {question}")
            print("-" * 50)

            # Agent 실행
            # messages 형식으로 질문을 전달합니다
            result = await agent.ainvoke({
                "messages": [{"role": "user", "content": question}]
            })

            # 중간 과정 출력 (Agent가 어떤 생각을 했는지)
            print("🔍 Agent의 처리 과정:")
            for msg in result["messages"]:
                msg_type = type(msg).__name__

                if msg_type == "HumanMessage":
                    print(f"  👤 사용자 입력: {msg.content}")
                elif msg_type == "AIMessage":
                    if msg.tool_calls:  # 도구를 호출하는 경우
                        for tc in msg.tool_calls:
                            print(f"  🤖 AI 결정: {tc['name']} 도구 사용 → 인자: {tc['args']}")
                    else:  # 최종 답변
                        print(f"  💬 최종 답변: {msg.content}")
                elif msg_type == "ToolMessage":
                    print(f"  🔧 도구 결과: {msg.content}")

            return result["messages"][-1].content

# 테스트 실행!
answer = await ask_agent("15 곱하기 7을 계산하고, 거기서 23을 빼줘")

C:\Users\vega2\AppData\Local\Temp\ipykernel_2740\2508852269.py:29: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(model, tools)


❓ 질문: 15 곱하기 7을 계산하고, 거기서 23을 빼줘
--------------------------------------------------
🔍 Agent의 처리 과정:
  👤 사용자 입력: 15 곱하기 7을 계산하고, 거기서 23을 빼줘
  🤖 AI 결정: multiply 도구 사용 → 인자: {'a': 15, 'b': 7}
  🤖 AI 결정: subtract 도구 사용 → 인자: {'a': 15, 'b': 23}
  🔧 도구 결과: [{'type': 'text', 'text': '105.0', 'id': 'lc_3a5ec7e8-8178-4ce9-8730-cdd7bec3aaff'}]
  🔧 도구 결과: [{'type': 'text', 'text': '-8.0', 'id': 'lc_7109bd37-feca-434c-9006-8556426c2c0a'}]
  💬 최종 답변: 15 곱하기 7은 105이고, 15에서 23을 빼면 -8입니다.


In [ ]:
# 다른 질문도 해봐요!
await ask_agent("100을 4로 나누고, 그 결과에 17을 더한 다음, 전체를 3으로 곱해줘")

C:\Users\vega2\AppData\Local\Temp\ipykernel_2740\2508852269.py:29: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(model, tools)


❓ 질문: 100을 4로 나누고, 그 결과에 17을 더한 다음, 전체를 3으로 곱해줘
--------------------------------------------------
🔍 Agent의 처리 과정:
  👤 사용자 입력: 100을 4로 나누고, 그 결과에 17을 더한 다음, 전체를 3으로 곱해줘
  🤖 AI 결정: divide 도구 사용 → 인자: {'a': 100, 'b': 4}
  🤖 AI 결정: add 도구 사용 → 인자: {'a': 17, 'b': 0}
  🔧 도구 결과: [{'type': 'text', 'text': '25.0', 'id': 'lc_3fe20664-433d-49a0-9e26-914995c8e7f1'}]
  🔧 도구 결과: [{'type': 'text', 'text': '17.0', 'id': 'lc_ec0be83c-a4c6-4ec1-b9cf-fb3ef61725a3'}]
  🤖 AI 결정: multiply 도구 사용 → 인자: {'a': 25, 'b': 3}
  🔧 도구 결과: [{'type': 'text', 'text': '75.0', 'id': 'lc_a62fd112-a700-437f-9350-6026a4cd077b'}]
  💬 최종 답변: 100을 4로 나눈 결과에 17을 더한 후, 전체를 3으로 곱한 결과는 75입니다.


'100을 4로 나눈 결과에 17을 더한 후, 전체를 3으로 곱한 결과는 75입니다.'

---
## STEP 5. 대화 이어가기 (멀티턴 대화)

이전 대화를 기억하면서 계속 이야기하는 방법입니다.

```
[ 멀티턴 동작 방식 ]

  messages = []  ← 대화 기록을 저장하는 리스트

  1번째 질문: "12 곱하기 8은?"
  messages = [
    {user: "12 곱하기 8은?"},
    {assistant: "96입니다"}    ← 대화 기록 추가
  ]

  2번째 질문: "거기서 30을 빼면?"  ← "거기서"가 96을 가리킴!
  messages = [
    {user: "12 곱하기 8은?"},
    {assistant: "96입니다"},
    {user: "거기서 30을 빼면?"},    ← 이전 대화가 있어서 문맥 파악 가능
    {assistant: "66입니다"}
  ]
```

> **💡 포인트**: 매번 `messages` 전체를 Agent에 전달해야 이전 대화를 기억합니다!

In [ ]:
import subprocess

async def multi_turn_chat():
    """대화 기록을 유지하며 여러 번 대화합니다"""

    # ✅ errlog=subprocess.DEVNULL : Windows + Jupyter 호환 필수 옵션
    async with stdio_client(server_params, errlog=subprocess.DEVNULL) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            tools = await load_mcp_tools(session)
            agent = create_react_agent(model, tools)

            # 📝 대화 기록을 저장할 리스트
            messages = []

            # 연속으로 할 질문들
            questions = [
                "12 곱하기 8은 얼마야?",
                "방금 나온 결과에서 30을 빼면?",   # '방금 나온 결과' = 96
                "그 결과를 2로 나눠줘",             # '그 결과' = 66
            ]

            print("=" * 50)
            print("💬 멀티턴 대화 시작")
            print("=" * 50)

            for i, question in enumerate(questions, 1):
                print(f"\n[{i}번째 대화]")
                print(f"👤 나: {question}")

                # 새 질문을 대화 기록에 추가
                messages.append({"role": "user", "content": question})

                # 전체 대화 기록을 Agent에 전달 (이전 문맥 포함!)
                result = await agent.ainvoke({"messages": messages})

                # AI 응답 추출
                ai_response = result["messages"][-1].content

                # AI 응답도 대화 기록에 추가 (다음 번 질문에서 참조)
                messages.append({"role": "assistant", "content": ai_response})

                print(f"🤖 AI: {ai_response}")
                print(f"   (현재 대화 기록 수: {len(messages)}개)")

            print("\n" + "=" * 50)
            print("✅ 대화 완료")

await multi_turn_chat()

C:\Users\vega2\AppData\Local\Temp\ipykernel_2740\1548082102.py:11: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(model, tools)


💬 멀티턴 대화 시작

[1번째 대화]
👤 나: 12 곱하기 8은 얼마야?
🤖 AI: 12 곱하기 8은 96입니다.
   (현재 대화 기록 수: 2개)

[2번째 대화]
👤 나: 방금 나온 결과에서 30을 빼면?
🤖 AI: 96에서 30을 빼면 66이 됩니다.
   (현재 대화 기록 수: 4개)

[3번째 대화]
👤 나: 그 결과를 2로 나눠줘
🤖 AI: 66을 2로 나누면 33이 됩니다.
   (현재 대화 기록 수: 6개)

✅ 대화 완료


---
## 🎓 전체 정리

### 배운 내용 요약

```
┌─────────────────────────────────────────────────────────┐
│                   LangChain + MCP 흐름                  │
│                                                         │
│  1. MCP 서버 만들기                                      │
│     FastMCP + @mcp.tool() 데코레이터                     │
│                                                         │
│  2. 연결 & 도구 로드                                     │
│     stdio_client → ClientSession → load_mcp_tools()     │
│                                                         │
│  3. Agent 만들기                                         │
│     create_react_agent(모델, 도구들)                      │
│                                                         │
│  4. 질문하기                                             │
│     agent.ainvoke({"messages": [...]})                  │
│                                                         │
└─────────────────────────────────────────────────────────┘
```

### 🪟 Windows + Jupyter 필수 패턴

```python
import subprocess

# 단일 서버: errlog=subprocess.DEVNULL 추가
async with stdio_client(server_params, errlog=subprocess.DEVNULL) as (read, write):
    ...

# 여러 서버: 각 서버 설정에 "errlog": subprocess.DEVNULL 추가
async with MultiServerMCPClient({
    "서버명": {
        "command": "python",
        "args": ["server.py"],
        "transport": "stdio",
        "errlog": subprocess.DEVNULL,  # ← 이게 핵심!
    }
}) as client:
    ...
```

### 다음 단계로 나아가기
- 🔍 [LangChain MCP Adapters 공식 문서](https://github.com/langchain-ai/langchain-mcp-adapters)
- 🌐 [MCP 공식 서버 목록](https://github.com/modelcontextprotocol/servers) - 파일시스템, GitHub, Slack 등 다양한 서버
- 📚 [LangGraph 문서](https://langchain-ai.github.io/langgraph/) - 더 복잡한 Agent 만들기